# 02 — Results: ROC, calibration, stage, explanations

Reads finished runs from `reports/` (run `python -m pancreasignal.train ...` first). No model is evaluated here; every performance number comes from the out-of-fold files. Figures used in the README are saved to `docs/figures/`.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import shap
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.calibration import calibration_curve

from pancreasignal.config import PROJECT_ROOT, load_config
from pancreasignal.evaluation.metrics import expected_calibration_error
from pancreasignal.explain import logreg_coefficients, oof_shap_values
from pancreasignal.reporting import TASKS, VARIANTS, get_run, load_runs

sns.set_theme(style="whitegrid", context="notebook")
FIG_DIR = PROJECT_ROOT / "docs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
COLORS = {"age_only": "#8C8C8C", "logreg": "#C44E52", "logreg_calibrated": "#DD8452",
          "xgboost": "#4C72B0", "xgboost_calibrated": "#64B5CD", "ca199": "#55A868"}
TASK = "pdac_vs_benign"   # the clinically hard comparison; change to look at another task

runs = load_runs()

def oof(variant, task=TASK):
    """Out-of-fold predictions of the newest run, only the patients that were scored."""
    return pd.read_csv(get_run(runs, task, variant)["oof_path"]).dropna(subset=["prob"])

runs[["task", "variant", "split", "run_id"]]

## 1. AUROC of every model on every task
Pooled out-of-fold AUROC. `pdac_vs_control` is random 5-fold (all controls come from one hospital); the other two are leave-one-center-out.

In [ ]:
auroc = pd.DataFrame({
    TASKS[t]: {VARIANTS[v]: get_run(runs, t, v)["results"]["model_all"]["auroc"] for v in VARIANTS}
    for t in TASKS
}).round(3)
auroc

## 2. ROC curves
Left: all scored patients. Right: only patients with a CA19-9 value, so the urine model and CA19-9 are compared on exactly the same people. The dotted lines mark 90% and 95% specificity (false-positive rate 0.10 and 0.05): the region that matters for screening.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# left: models on all scored patients
for v in ["age_only", "logreg", "xgboost"]:
    d = oof(v)
    fpr, tpr, _ = roc_curve(d.y, d.prob)
    axes[0].plot(fpr, tpr, color=COLORS[v], lw=2, label=f"{VARIANTS[v]} ({roc_auc_score(d.y, d.prob):.3f})")
axes[0].set_title(f"{TASKS[TASK]}: all scored patients (n={len(oof('logreg'))})")

# right: same rows as CA19-9
d = oof("logreg").dropna(subset=["ca199"])
for label, score, color in [("Logistic regression", d.prob, COLORS["logreg"]),
                            ("Plasma CA19-9", d.ca199, COLORS["ca199"])]:
    fpr, tpr, _ = roc_curve(d.y, score)
    axes[1].plot(fpr, tpr, color=color, lw=2, label=f"{label} ({roc_auc_score(d.y, score):.3f})")
axes[1].set_title(f"Same patients as CA19-9 (n={len(d)})")

for ax in axes:
    ax.plot([0, 1], [0, 1], color="#BBBBBB", lw=1, ls="--")
    for x in (0.05, 0.10):
        ax.axvline(x, color="#999999", lw=1, ls=":")
    ax.set(xlabel="False-positive rate (1 - specificity)", ylabel="Sensitivity", xlim=(0, 1), ylim=(0, 1))
    ax.legend(loc="lower right", title="AUROC")
plt.tight_layout()
fig.savefig(FIG_DIR / f"roc_{TASK}.png", dpi=150, bbox_inches="tight")
plt.show()

**What this shows.**
- The logistic regression (red) sits above XGBoost and far above age alone in the low false-positive region (left of the dotted lines), which is where a screening test would operate.
- On the same patients, CA19-9 (green) is much better at high specificity: at a 5% false-positive rate it finds roughly three quarters of the cancers, the urine model about a quarter. In this data the urine panel is not a replacement for CA19-9.

## 3. Calibration
If the model says 30%, about 30% of those patients should have PDAC. Points on the diagonal = well calibrated. ECE = average distance from the diagonal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, pair in zip(axes, [("logreg", "logreg_calibrated"), ("xgboost", "xgboost_calibrated")]):
    for v in pair:
        d = oof(v)
        frac_pos, mean_pred = calibration_curve(d.y, d.prob, n_bins=10, strategy="uniform")
        ece = expected_calibration_error(d.y, d.prob)
        ax.plot(mean_pred, frac_pos, "o-", color=COLORS[v], lw=2, label=f"{VARIANTS[v]} (ECE {ece:.3f})")
    ax.plot([0, 1], [0, 1], color="#BBBBBB", lw=1, ls="--")
    ax.set(xlabel="Predicted risk", ylabel="Observed PDAC rate", xlim=(0, 1), ylim=(0, 1))
    ax.legend(loc="upper left", fontsize=9)
axes[0].set_title(f"{TASKS[TASK]}: logistic regression")
axes[1].set_title(f"{TASKS[TASK]}: XGBoost")
plt.tight_layout()
fig.savefig(FIG_DIR / f"calibration_{TASK}.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Calibration per held-out hospital: average predicted risk vs real PDAC rate.
# This is the part that calibration fitted on OTHER hospitals cannot fix.
per_center = pd.concat({
    VARIANTS[v]: oof(v).groupby("center").agg(predicted=("prob", "mean"), observed=("y", "mean"), n=("y", "size"))
    for v in ["age_only", "logreg", "logreg_calibrated", "xgboost"]
}).round(2)
per_center

**What this shows.**
- Logistic regression is already close to the diagonal; isotonic calibration does not improve it.
- XGBoost's raw probabilities are far off (too low around 0.2–0.4, too high near 0.9). Isotonic calibration roughly halves its ECE.
- The per-hospital table shows the error that remains for every model: predicted risk is too high at BPTB and too low at LIV, because the PDAC rate differs by hospital. The calibrator is fitted on the training hospitals, so it cannot know the PDAC rate of a new one. Before real use the model would need recalibrating on local data.

## 4. Sensitivity by stage
At the 90% / 95% specificity threshold, what share of early (I–II) and late (III–IV) cancers is caught? Screening only helps if early cancers are found.

In [ ]:
rows = []
for t in TASKS:
    for v in VARIANTS:
        for spec, groups in get_run(runs, t, v)["results"]["sens_by_stage"].items():
            for g, val in groups.items():
                rows.append({"task": TASKS[t], "model": VARIANTS[v], "specificity": spec[4:] + "%",
                             "stage": g, "n_pdac": val["n_pdac"], "sensitivity": val["sensitivity"]})
stage = pd.DataFrame(rows)
stage.pivot_table(index=["task", "model"], columns=["specificity", "stage"], values="sensitivity").round(2)

In [ ]:
# Where do early vs late cases come from? (a center effect could hide behind a stage effect)
d = oof("logreg")
d = d[d.y == 1].assign(stage_group=lambda t: np.where(t.stage.str.startswith(("III", "IV")), "III-IV", "I-II"))
pd.crosstab(d.center, d.stage_group, margins=True)

**What this shows.**
- For logistic regression, early (I–II) cancers are caught at least as often as late (III–IV) ones. That is encouraging for screening, but:
  - the early group is mostly stage IIB; only 16 patients are stage I;
  - each group has about 100 patients, so differences below ~0.1 are within noise;
  - early cases come mostly from LIV and late ones more from BPTB, so a stage comparison is partly a hospital comparison.

## 5. What does XGBoost rely on? (SHAP, out-of-fold)
Each patient is explained by the fold model that never saw them. One dot = one patient; position = how much that feature pushed the prediction towards PDAC (+) or away (−); colour = the feature value (red high, blue low).

In [ ]:
cfg = load_config("configs/urine.yaml")
cfg["task"], cfg["model"]["name"] = TASK, "xgboost"
X_shap, shap_values = oof_shap_values(cfg)

importance = shap_values.abs().mean().sort_values(ascending=False).rename("mean |SHAP|").round(3)
display(importance.to_frame())

shap.summary_plot(shap_values.to_numpy(), X_shap, show=False, plot_size=(8, 4))
plt.title(f"XGBoost, {TASKS[TASK]} (out-of-fold SHAP)")
plt.tight_layout()
plt.savefig(FIG_DIR / f"shap_{TASK}.png", dpi=150, bbox_inches="tight")
plt.show()

**What this shows.**
- XGBoost relies on age the most, then LYVE1 and REG1B; TFF1 less, sex hardly at all.
- High LYVE1 and high age (red dots on the right) push the prediction towards PDAC, as expected.
- Leaning on age may be part of why XGBoost transfers worse between hospitals: the age-only model is the one hit hardest by the hospital shift (section 3 and RESULTS.md).

## 6. Logistic regression coefficients
Fitted once on all rows of the task (interpretation only). Features are standardised inside the Pipeline, so each coefficient is the change in log-odds of PDAC for +1 standard deviation; odds ratio = exp(coefficient).

In [ ]:
cfg = load_config("configs/urine.yaml")
cfg["task"], cfg["model"]["name"] = TASK, "logreg"
logreg_coefficients(cfg).round(3)

**What this shows.**
- LYVE1 is the strongest feature: +1 SD of log1p(LYVE1 / creatinine) multiplies the odds of PDAC by about 2.5. Age is second, then REG1B and TFF1; sex adds almost nothing.
- Same top features as SHAP, except that XGBoost puts age first. Both agree that the urine proteins carry signal beyond age, LYVE1 most of all.
- The original study built its score from the same three proteins plus creatinine and age; compare with the feature ranking in the paper itself before claiming agreement.

## Summary

- **Keep logistic regression:** best AUROC on unseen hospitals, close to calibrated, easy to explain.
- **XGBoost** is worse on unseen hospitals and needs calibration; on patients from the training hospitals it ties.
- **Biomarkers beat age at BPTB, not at LIV** (all benign patients at LIV have pancreatitis).
- **Risk scores do not transfer between hospitals** without local recalibration.
- **Plasma CA19-9 is better on the same patients.** The next useful question: does the urine panel add anything on top of CA19-9?